In [2]:
!pip install -q hmmlearn statsmodels openpyxl seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 3.2 MB/s eta 0:00:00


In [3]:
# %% [markdown]
# # PalmSeq — a bioinformatics framework for longitudinal Palmyra (Borassus flabellifer) health data
#
# Dataset : Velanai Palmyra dataset (50 trees x 9 bi-monthly visits, Mar 2021 – Jul 2022)
#
# Core idea: treat each tree's visit-by-visit health history as a biological *sequence*
# and each measured trait as a *gene*, then reuse classic bioinformatics machinery:
#
# | Module | Bioinformatics analogue | Question answered |
# |---|---|---|
# | M1 | Sequence encoding (alphabet) | Encode each visit as a stress "residue" H/L/M/S |
# | M2 | BLOSUM-style log-odds matrix | Which health states substitute for each other inside a tree? |
# | M3 | Needleman–Wunsch + Feng–Doolittle distance + UPGMA | "Phenotypic phylogeny" of trees → health clades |
# | M4 | PERMANOVA | Do clades follow land use / soil / salt / coast distance? |
# | M5 | Markov chain + k-mer motif enrichment | Transition dynamics; motifs enriched in coastal trees |
# | M6 | Hidden Markov Model (Viterbi) | Latent physiological states behind the scores |
# | M7 | WGCNA-style co-variation network | Trait modules ("eigentraits") and their environmental drivers |
# | M8 | EWAS (environment-wide association, GEE) | Which exposures associate with stress / pest / disease? |
# | M9 | limma-style empirical-Bayes differential growth | Do coastal palms grow differently? (volcano plot) |
# | M10 | Leakage-safe predictive validation | Can we forecast next-visit canopy stress? |
#
# Run:  python palmseq_pipeline.py --data Velanai_Palmera_Adata_Set_PDB.xlsx --out outputs
# Colab: !pip install hmmlearn statsmodels lightgbm openpyxl  then run the cells top to bottom.

# %%
!pip install hmmlearn statsmodels lightgbm openpyxl

# %%
import argparse, json, os, sys, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import digamma, polygamma
from scipy.optimize import brentq
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score, roc_auc_score, balanced_accuracy_score, brier_score_loss
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
import statsmodels.formula.api as smf
from hmmlearn.hmm import CategoricalHMM

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)

def in_notebook():
    return "ipykernel" in sys.modules

if in_notebook():
    DATA, OUT = "/content/Velanai Palmera Adata Set PDB.xlsx", "outputs"
else:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="Velanai_Palmera_Adata_Set_PDB.xlsx")
    ap.add_argument("--out", default="outputs")
    a = ap.parse_args(); DATA, OUT = a.data, a.out
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.family": "DejaVu Sans"})
STATE_COL = {"H": "#2E7D32", "L": "#9CCC65", "M": "#F9A825", "S": "#C62828"}
SUMMARY = {}

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.close()

# %% [markdown]
# ## 0. Load, clean and engineer variables

# %%
raw = pd.read_excel(DATA)
df = raw.rename(columns={
    "Distance from Coast (m)": "CoastDist_m", "Land Use Type": "LandUse", "Soil_Type": "Soil",
    "Tree Height (m)": "Height_m", "Trunk Girth (cm)": "Girth_cm", "DBH (cm)": "DBH_cm",
    "Crown Diameter (m)": "Crown_m", "Live Leaves Count": "LiveLeaves", "Dead Leaves Count": "DeadLeaves",
    "New Leaves Count": "NewLeaves", "Canopy  Health Score 03": "Canopy", "Leaf Yellowing Score 03": "Yellowing",
    "Leaf Drying Score 03": "Drying", "Pest Present": "Pest", "Pest Severity 03": "PestSev",
    "Disease Present": "Disease", "Disease Severity 03": "DiseaseSev", "Flowering Status": "Flowering",
    "Fruiting Status": "Fruiting", "Soil Moisture": "SoilMoisture", "Soil pH": "SoilPH", "Soil EC dS (m)": "SoilEC",
    "Soil Organic Carbon pct": "SOC", "Rainfall Previous 2 Months mm": "Rain2m", "Mean Temperature C": "TempC",
    "Relative Humidity pct": "RH", "Flooding Waterlogging": "Flooding", "Salt Exposure": "SaltExposure",
    "Human Utilization": "HumanUse", "Mechanical Damage": "MechDamage", "Elevation_m": "Elevation_m"})
df = df.drop(columns=["Remarks"])  # text derived from the scores -> label leakage
df = df.sort_values(["Tree_ID", "Observation_Date"]).reset_index(drop=True)
for c in ["Pest", "Disease", "Flowering", "Fruiting", "Flooding", "HumanUse", "MechDamage"]:
    df[c] = (df[c] == "Yes").astype(int)
df["SaltOrd"] = df["SaltExposure"].map({"Low": 0, "Moderate": 1, "High": 2})
df["MoistOrd"] = df["SoilMoisture"].map({"Low": 0, "Moderate": 1, "High": 2})
df["Visit"] = df.groupby("Tree_ID").cumcount()
df["Years"] = (df["Observation_Date"] - df["Observation_Date"].min()).dt.days / 365.25
df["WetSeason"] = df["Observation_Date"].dt.month.isin([11, 1]).astype(int)  # NE-monsoon visits
# Composite leaf-stress index (0–9): yellowing + drying + canopy deficit
df["StressIdx"] = df["Yellowing"] + df["Drying"] + (3 - df["Canopy"])
df["PoorCanopy"] = (df["Canopy"] < 3).astype(int)
df["Coastal"] = (df["SaltOrd"] >= 1).astype(int)

trees = df.groupby("Tree_ID").first()
T, V = df.Tree_ID.nunique(), df.Visit.nunique()
SUMMARY["n_trees"], SUMMARY["n_visits"], SUMMARY["n_rows"] = int(T), int(V), len(df)
print(f"{T} trees x {V} visits = {len(df)} observations")

# descriptive table
desc = df.describe().T[["mean", "std", "min", "max"]].round(3)
desc.to_csv(os.path.join(TAB, "T0_descriptives.csv"))

# %% [markdown]
# ## M1. Sequence encoding — the "health alphabet"
# Stress index → residue: H (0–1, healthy), L (2–3, light), M (4–5, moderate), S (≥6, severe).
# Pest / disease are kept as separate annotation tracks (like PTMs on a protein).

# %%
ALPHA = ["H", "L", "M", "S"]
def residue(s):
    return "H" if s <= 1 else "L" if s <= 3 else "M" if s <= 5 else "S"
df["Residue"] = df["StressIdx"].apply(residue)
seqs = df.groupby("Tree_ID")["Residue"].apply("".join)
pest_track = df.groupby("Tree_ID")["Pest"].apply(lambda x: "".join("p" if v else "." for v in x))
dis_track = df.groupby("Tree_ID")["Disease"].apply(lambda x: "".join("d" if v else "." for v in x))
with open(os.path.join(TAB, "palmyra_health_sequences.fasta"), "w") as f:
    for t in seqs.index:
        f.write(f">{t}|{trees.loc[t,'LandUse']}|{trees.loc[t,'Soil']}|salt={trees.loc[t,'SaltExposure']}"
                f"|coast={trees.loc[t,'CoastDist_m']}m\n{seqs[t]}\n")
SUMMARY["residue_freq"] = df["Residue"].value_counts(normalize=True).round(3).to_dict()

# Sequence-logo-like positional frequency plot (per visit)
pos = pd.crosstab(df["Visit"], df["Residue"], normalize="index")[ALPHA]
fig, ax = plt.subplots(figsize=(7, 3))
bottom = np.zeros(V)
for a_ in ALPHA:
    ax.bar(range(V), pos[a_], bottom=bottom, color=STATE_COL[a_], label=a_, width=0.85)
    bottom += pos[a_].values
ax.set_xticks(range(V)); ax.set_xticklabels(df.Observation_Date.dt.strftime("%b-%y").unique(), rotation=45)
ax.set_ylabel("Residue frequency"); ax.set_title("Positional health-residue profile (sequence-logo analogue)")
ax.legend(ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.18), loc="upper center")
savefig("F1_positional_residue_profile.png")

# Sequence "MSA" heat-map ordered later by phylogeny (drawn in M3)

# %% [markdown]
# ## M2. PALM-BLOSUM — empirically learned substitution matrix
# BLOSUM logic: residues observed in the same *block* (here: the same tree) are "aligned".
# q_ij = observed pair frequency, e_ij = p_i p_j (2 p_i p_j off-diagonal); s_ij = round(2·log2(q_ij/e_ij)).

# %%
def learn_blosum(seq_list, alpha=ALPHA, pseudo=0.5):
    idx = {a: i for i, a in enumerate(alpha)}
    C = np.full((len(alpha), len(alpha)), pseudo)
    for s in seq_list:
        for i, j in itertools.combinations(range(len(s)), 2):
            a, b = idx[s[i]], idx[s[j]]
            C[a, b] += 1; C[b, a] += 1 if a != b else 0
    q = np.triu(C); q = q / q.sum()
    p = np.array([q[i, i] + 0.5 * (q[i, :].sum() - q[i, i]) + 0.5 * (q[:, i].sum() - q[i, i]) for i in range(len(alpha))])
    S = np.zeros_like(q)
    for i in range(len(alpha)):
        for j in range(i, len(alpha)):
            e = p[i] * p[j] if i == j else 2 * p[i] * p[j]
            S[i, j] = S[j, i] = np.round(2 * np.log2(q[i, j] / e))
    return pd.DataFrame(S.astype(int), index=alpha, columns=alpha), p

BLOSUM, bg = learn_blosum(seqs.tolist())
BLOSUM.to_csv(os.path.join(TAB, "T1_PALM_BLOSUM.csv"))
print("PALM-BLOSUM:\n", BLOSUM)
fig, ax = plt.subplots(figsize=(3.6, 3.1))
sns.heatmap(BLOSUM, annot=True, fmt="d", cmap="RdBu_r", center=0, cbar_kws={"label": "log-odds (half-bits)"}, ax=ax)
ax.set_title("PALM-BLOSUM substitution matrix"); savefig("F2_PALM_BLOSUM.png")

# %% [markdown]
# ## M3. Pairwise global alignment → Feng–Doolittle distance → UPGMA "phenotypic phylogeny"

# %%
GAP = -4
def nw_score(a, b, S=BLOSUM, gap=GAP):
    n, m = len(a), len(b); F = np.zeros((n + 1, m + 1))
    F[:, 0] = gap * np.arange(n + 1); F[0, :] = gap * np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            F[i, j] = max(F[i-1, j-1] + S.at[a[i-1], b[j-1]], F[i-1, j] + gap, F[i, j-1] + gap)
    return F[n, m]

def nw_align(a, b, S=BLOSUM, gap=GAP):  # with traceback, for illustration
    n, m = len(a), len(b); F = np.zeros((n + 1, m + 1))
    F[:, 0] = gap * np.arange(n + 1); F[0, :] = gap * np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            F[i, j] = max(F[i-1, j-1] + S.at[a[i-1], b[j-1]], F[i-1, j] + gap, F[i, j-1] + gap)
    i, j, A, B = n, m, "", ""
    while i > 0 or j > 0:
        if i > 0 and j > 0 and F[i, j] == F[i-1, j-1] + S.at[a[i-1], b[j-1]]:
            A, B, i, j = a[i-1] + A, b[j-1] + B, i - 1, j - 1
        elif i > 0 and F[i, j] == F[i-1, j] + gap:
            A, B, i = a[i-1] + A, "-" + B, i - 1
        else:
            A, B, j = "-" + A, b[j-1] + B, j - 1
    return A, B, F[n, m]

ids = seqs.index.tolist(); N = len(ids)
SC = np.zeros((N, N))
for i in range(N):
    for j in range(i, N):
        SC[i, j] = SC[j, i] = nw_score(seqs[ids[i]], seqs[ids[j]])

def s_rand(a, b, k=30):
    return np.mean([nw_score("".join(rng.permutation(list(a))), "".join(rng.permutation(list(b)))) for _ in range(k)])

D = np.zeros((N, N))
for i in range(N):
    for j in range(i + 1, N):
        sr = s_rand(seqs[ids[i]], seqs[ids[j]], k=10)
        smax = 0.5 * (SC[i, i] + SC[j, j])
        x = (SC[i, j] - sr) / max(smax - sr, 1e-6)
        D[i, j] = D[j, i] = -np.log(np.clip(x, 1e-3, 1.0))   # Feng–Doolittle distance
Dm = pd.DataFrame(D, index=ids, columns=ids); Dm.to_csv(os.path.join(TAB, "T2_FengDoolittle_distance.csv"))

Z = linkage(squareform(D, checks=False), method="average")  # UPGMA
sil = {k: silhouette_score(D, fcluster(Z, k, "maxclust"), metric="precomputed") for k in range(2, 7)}
K = max(sil, key=sil.get)
clade = pd.Series(fcluster(Z, K, "maxclust"), index=ids, name="Clade")
trees["Clade"] = clade
SUMMARY["upgma_silhouette"] = {int(k): round(v, 3) for k, v in sil.items()}; SUMMARY["n_clades"] = int(K)

# Clade profile table
cp = trees.groupby("Clade").agg(n=("LandUse", "size"), coast_m=("CoastDist_m", "mean"),
                                 salt=("SaltOrd", "mean"), elev=("Elevation_m", "mean"))
cp["mean_stress"] = df.merge(clade, left_on="Tree_ID", right_index=True).groupby("Clade")["StressIdx"].mean()
cp["consensus_seq"] = [ "".join(pd.DataFrame([list(seqs[t]) for t in clade[clade == c].index]).mode().iloc[0])
                        for c in cp.index]
cp.round(2).to_csv(os.path.join(TAB, "T3_clade_profiles.csv")); print(cp.round(2))

# Dendrogram + aligned "MSA" panel
fig = plt.figure(figsize=(9, 9)); gs = fig.add_gridspec(1, 3, width_ratios=[2.2, 2, 0.6], wspace=0.02)
ax0 = fig.add_subplot(gs[0]); dn = dendrogram(Z, orientation="left", labels=ids, ax=ax0, color_threshold=Z[-(K-1), 2],
                                              above_threshold_color="#888", leaf_font_size=6)
ax0.set_title(f"UPGMA phenotypic phylogeny (k={K} clades)"); ax0.set_xlabel("Feng–Doolittle distance")
order = dn["ivl"][::-1]
ax1 = fig.add_subplot(gs[1])
cmap = matplotlib.colors.ListedColormap([STATE_COL[a] for a in ALPHA])
M = np.array([[ALPHA.index(c) for c in seqs[t]] for t in order])
ax1.imshow(M, aspect="auto", cmap=cmap, vmin=-0.5, vmax=3.5)
for r, t in enumerate(order):
    for c in range(V):
        mark = ("p" if pest_track[t][c] == "p" else "") + ("d" if dis_track[t][c] == "d" else "")
        if mark: ax1.text(c, r, mark, ha="center", va="center", fontsize=5, color="k")
ax1.set_yticks([]); ax1.set_xticks(range(V)); ax1.set_xticklabels(range(1, V + 1)); ax1.set_xlabel("Visit")
ax1.set_title("Health 'MSA' (p=pest, d=disease)")
ax2 = fig.add_subplot(gs[2]); salt = trees.loc[order, "SaltOrd"].values[:, None]
ax2.imshow(salt, aspect="auto", cmap="Blues", vmin=-0.5, vmax=2); ax2.set_yticks([]); ax2.set_xticks([0])
ax2.set_xticklabels(["Salt"])
handles = [matplotlib.patches.Patch(color=STATE_COL[a], label=a) for a in ALPHA]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False)
savefig("F3_phylogeny_MSA.png")

# Example alignment of two most distant trees in different clades
i, j = np.unravel_index(np.argmax(D), D.shape)
A_, B_, sc_ = nw_align(seqs[ids[i]], seqs[ids[j]])
SUMMARY["example_alignment"] = {"pair": [ids[i], ids[j]], "A": A_, "B": B_, "score": float(sc_)}

# %% [markdown]
# ## M4. PERMANOVA — are the health clades explained by site factors?

# %%
def permanova(Dmat, groups, n_perm=4999):
    groups = np.asarray(groups); n = len(groups); D2 = Dmat ** 2
    def F(g):
        ug = np.unique(g); a = len(ug)
        sst = D2[np.triu_indices(n, 1)].sum() / n
        ssw = sum(D2[np.ix_(g == u, g == u)][np.triu_indices((g == u).sum(), 1)].sum() / (g == u).sum() for u in ug)
        ssa = sst - ssw
        return (ssa / (a - 1)) / (ssw / (n - a)), ssa / sst
    f0, r2 = F(groups)
    perm = np.array([F(rng.permutation(groups))[0] for _ in range(n_perm)])
    return f0, r2, (np.sum(perm >= f0) + 1) / (n_perm + 1)

trees["CoastTertile"] = pd.qcut(trees["CoastDist_m"], 3, labels=["near", "mid", "far"])
pm = []
for fac in ["LandUse", "Soil", "SaltExposure", "CoastTertile"]:
    f0, r2, p = permanova(D, trees.loc[ids, fac].astype(str).values, n_perm=1999)
    pm.append({"factor": fac, "pseudo_F": round(f0, 3), "R2": round(r2, 3), "p_perm": round(p, 4)})
pm = pd.DataFrame(pm); pm["q_BH"] = multipletests(pm.p_perm, method="fdr_bh")[1].round(4)
pm.to_csv(os.path.join(TAB, "T4_PERMANOVA.csv"), index=False); print(pm)

# %% [markdown]
# ## M5. Markov transitions and k-mer motif enrichment (coastal vs inland trees)

# %%
trans = pd.crosstab(df.groupby("Tree_ID")["Residue"].shift().dropna(),
                    df.loc[df.Visit > 0, "Residue"], normalize="index").reindex(index=ALPHA, columns=ALPHA).fillna(0)
trans.round(3).to_csv(os.path.join(TAB, "T5_transition_matrix.csv"))
fig, ax = plt.subplots(figsize=(3.6, 3.1))
sns.heatmap(trans, annot=True, fmt=".2f", cmap="Greens", ax=ax, cbar=False)
ax.set_xlabel("State at t+1"); ax.set_ylabel("State at t"); ax.set_title("First-order Markov transitions")
savefig("F4_transition_matrix.png")

# Markov-order test: does state at t predict t+1 beyond chance? (chi-square)
ct = pd.crosstab(df.groupby("Tree_ID")["Residue"].shift().dropna(), df.loc[df.Visit > 0, "Residue"])
chi = stats.chi2_contingency(ct); SUMMARY["markov_chi2_p"] = float(chi[1])
# same test for the pest track
pt = pd.crosstab(df.groupby("Tree_ID")["Pest"].shift().dropna(), df.loc[df.Visit > 0, "Pest"])
SUMMARY["pest_persistence_fisher_p"] = float(stats.fisher_exact(pt.values)[1])

def kmers(s, k): return {s[i:i+k] for i in range(len(s) - k + 1)}
motif_rows = []
for k in (2, 3):
    allk = sorted(set().union(*[kmers(s, k) for s in seqs]))
    for km in allk:
        has = np.array([km in kmers(seqs[t], k) for t in ids])
        coast = trees.loc[ids, "Coastal"].values == 1
        tab = np.array([[np.sum(has & coast), np.sum(~has & coast)], [np.sum(has & ~coast), np.sum(~has & ~coast)]])
        odds, p = stats.fisher_exact(tab)
        motif_rows.append({"k": k, "motif": km, "coastal_with": tab[0, 0], "coastal_n": coast.sum(),
                           "inland_with": tab[1, 0], "inland_n": (~coast).sum(),
                           "odds_ratio": odds, "p": p})
motifs = pd.DataFrame(motif_rows)
motifs["q_BH"] = multipletests(motifs.p, method="fdr_bh")[1]
motifs = motifs.sort_values("p"); motifs.round(4).to_csv(os.path.join(TAB, "T6_kmer_motif_enrichment.csv"), index=False)
print(motifs.head(10))

# %% [markdown]
# ## M6. Hidden Markov Model on the residue alphabet (CpG-island-style HMM)
# A categorical HMM emits H/L/M/S residues from hidden physiological states. The number of states is chosen
# by BIC (10 random restarts each). Viterbi decoding labels every visit; trees that ever enter the most
# stressed hidden state are tested for coastal enrichment.
# (A Gaussian HMM on the raw ordinal scores was tried first and collapsed onto the discrete canopy levels.)

# %%
RES_CODE = {a: i for i, a in enumerate(ALPHA)}
df["ResCode"] = df["Residue"].map(RES_CODE)

def fit_hmm(frame, n_states, seed=SEED, restarts=10):
    X = frame["ResCode"].values.reshape(-1, 1); lengths = frame.groupby("Tree_ID").size().values
    best = None
    for r in range(restarts):
        m = CategoricalHMM(n_components=n_states, n_iter=500, random_state=seed + r, n_features=len(ALPHA))
        m.fit(X, lengths); ll = m.score(X, lengths)
        if best is None or ll > best[1]: best = (m, ll)
    m = best[0]
    # order hidden states by expected emitted stress (0=H ... 3=S)
    order = np.argsort(m.emissionprob_ @ np.arange(len(ALPHA)))
    return m, order, best[1]

bic = {}
for ns in (2, 3, 4):
    m, _, ll = fit_hmm(df, ns)
    npar = ns * (ns - 1) + (ns - 1) + ns * (len(ALPHA) - 1)
    bic[ns] = -2 * ll + npar * np.log(len(df))
NS = min(bic, key=bic.get); SUMMARY["hmm_bic"] = {int(k): round(v, 1) for k, v in bic.items()}; SUMMARY["hmm_states"] = int(NS)
hmm, order_s, _ = fit_hmm(df, NS)
lengths = df.groupby("Tree_ID").size().values
raw_state = hmm.predict(df["ResCode"].values.reshape(-1, 1), lengths)
relabel = {old: new for new, old in enumerate(order_s)}
df["HMMState"] = [relabel[s] for s in raw_state]
ZN = [f"Z{i}" for i in range(NS)]
A = pd.DataFrame(hmm.transmat_[np.ix_(order_s, order_s)], index=ZN, columns=ZN)
E = pd.DataFrame(hmm.emissionprob_[order_s], index=ZN, columns=ALPHA)
A.round(3).to_csv(os.path.join(TAB, "T7_HMM_transitions.csv")); E.round(3).to_csv(os.path.join(TAB, "T7_HMM_emissions.csv"))
emis = df.groupby("HMMState")[["Yellowing", "Drying", "Canopy", "StressIdx", "Pest", "Disease", "SoilEC", "Rain2m"]].mean()
emis["n_obs"] = df["HMMState"].value_counts().sort_index()
emis.round(3).to_csv(os.path.join(TAB, "T7_HMM_state_profiles.csv")); print(E.round(2)); print(emis.round(2))
# coastal enrichment of trees that ever enter the top-stress hidden state
ever_top = df.groupby("Tree_ID")["HMMState"].max().eq(NS - 1)
tab = pd.crosstab(trees.loc[ids, "Coastal"], ever_top.loc[ids])
SUMMARY["hmm_top_state_trees"] = int(ever_top.sum())
SUMMARY["hmm_top_state_coastal_fisher"] = {"table": tab.values.tolist(),
                                           "OR_p": [float(x) for x in stats.fisher_exact(tab.values)]}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), gridspec_kw={"width_ratios": [1, 1.2, 1.6]})
sns.heatmap(A, annot=True, fmt=".2f", cmap="Purples", cbar=False, ax=axes[0])
axes[0].set_title("Transition matrix"); axes[0].set_xlabel("to"); axes[0].set_ylabel("from")
sns.heatmap(E, annot=True, fmt=".2f", cmap="Oranges", cbar=False, ax=axes[1]); axes[1].set_title("Emission probabilities")
occ = pd.crosstab(df["Visit"], df["HMMState"], normalize="index")
occ.columns = [f"Z{c}" for c in occ.columns]
occ.plot.area(ax=axes[2], color=["#66BB6A", "#FFA726", "#EF5350", "#8E24AA"][:NS], alpha=0.85)
axes[2].set_xticks(range(V)); axes[2].set_xticklabels(df.Observation_Date.dt.strftime("%b-%y").unique(), rotation=45)
axes[2].set_ylabel("Share of trees"); axes[2].set_title("Hidden-state occupancy (Viterbi)")
axes[2].legend(frameon=False, fontsize=7)
savefig("F5_HMM.png")

# %% [markdown]
# ## M7. WGCNA-style trait co-variation network
# Traits = "genes"; environment = "clinical traits". Within-tree centring removes stable between-tree
# differences in size so modules reflect co-varying *changes*.

# %%
GENES = ["Height_m", "Girth_cm", "DBH_cm", "Crown_m", "LiveLeaves", "DeadLeaves", "NewLeaves", "Canopy",
         "Yellowing", "Drying", "PestSev", "DiseaseSev", "Flowering", "Fruiting"]
ENV = ["SoilEC", "SoilPH", "SOC", "MoistOrd", "Rain2m", "TempC", "RH", "SaltOrd", "CoastDist_m", "Elevation_m",
       "HumanUse", "MechDamage", "WetSeason"]
G = df[GENES].astype(float)
Gc = G - df.groupby("Tree_ID")[GENES].transform("mean")   # within-tree centred
Gc[["Height_m", "Girth_cm", "DBH_cm", "Crown_m"]] = G[["Height_m", "Girth_cm", "DBH_cm", "Crown_m"]]  # keep size between-tree
C = Gc.corr(method="spearman").abs().values

def sft_fit(Cm, beta):
    k = (Cm ** beta).sum(1) - 1
    h, e = np.histogram(k, bins=6); x = (e[:-1] + e[1:]) / 2; m = h > 0
    if m.sum() < 3: return np.nan
    r = stats.linregress(np.log10(x[m]), np.log10(h[m] / h.sum()))
    return -np.sign(r.slope) * r.rvalue ** 2
sft = {b: sft_fit(C, b) for b in range(1, 13)}
BETA = next((b for b in sft if sft[b] >= 0.8), max(sft, key=lambda b: np.nan_to_num(sft[b], nan=-9)))
SUMMARY["wgcna_beta"] = int(BETA)
Adj = C ** BETA; np.fill_diagonal(Adj, 0)
L = Adj @ Adj; k = Adj.sum(1)
TOM = (L + Adj) / (np.minimum.outer(k, k) + 1 - Adj); np.fill_diagonal(TOM, 1)
Zt = linkage(squareform(1 - TOM, checks=False), "average")
mods = fcluster(Zt, t=0.9 * Zt[:, 2].max(), criterion="distance")
# merge singleton modules into "grey"
modser = pd.Series(mods, index=GENES); counts = modser.value_counts()
modser = modser.apply(lambda m: f"M{m}" if counts[m] > 1 else "grey")
modser.to_csv(os.path.join(TAB, "T8_trait_modules.csv"), header=["module"])
print(modser)

Zs = StandardScaler().fit_transform(Gc)
ME = {}
for mname in sorted(set(modser) - {"grey"}):
    cols = [GENES.index(g) for g in modser[modser == mname].index]
    u, s_, vt = np.linalg.svd(Zs[:, cols], full_matrices=False)
    pc = u[:, 0] * s_[0]
    if np.corrcoef(pc, Zs[:, cols].mean(1))[0, 1] < 0: pc = -pc
    ME[mname] = pc
ME = pd.DataFrame(ME)
r_me = pd.DataFrame(index=ME.columns, columns=ENV, dtype=float); p_me = r_me.copy()
for m_ in ME:
    for e_ in ENV:
        r, p = stats.spearmanr(ME[m_], df[e_]); r_me.loc[m_, e_] = r; p_me.loc[m_, e_] = p
q_me = pd.DataFrame(multipletests(p_me.values.ravel(), method="fdr_bh")[1].reshape(p_me.shape),
                    index=p_me.index, columns=p_me.columns)
r_me.round(3).to_csv(os.path.join(TAB, "T9_module_env_rho.csv")); q_me.to_csv(os.path.join(TAB, "T9_module_env_q.csv"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
dendrogram(Zt, labels=[f"{g} [{modser[g]}]" for g in GENES], orientation="left", ax=axes[0], leaf_font_size=7,
           color_threshold=0.9 * Zt[:, 2].max())
axes[0].set_title(f"Trait TOM dendrogram (soft power β={BETA})")
ann = r_me.round(2).astype(str) + np.where(q_me < 0.05, "*", "")
sns.heatmap(r_me.astype(float), annot=ann, fmt="", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=axes[1],
            annot_kws={"fontsize": 6}, cbar_kws={"label": "Spearman ρ"})
axes[1].set_title("Module eigentrait – environment correlation (* q<0.05)")
savefig("F6_WGCNA_modules.png")

# %% [markdown]
# ## M8. EWAS — environment-wide association study (GEE, exchangeable within-tree correlation)
# Outcomes: StressIdx (Gaussian), PoorCanopy, Pest, Disease (binomial). All adjusted for WetSeason
# (except the climate/season exposures themselves). BH-FDR across the full exposure × outcome grid.

# %%
EXPO = {"SoilEC": "c", "SoilPH": "c", "SOC": "c", "MoistOrd": "c", "Rain2m": "c", "TempC": "c", "RH": "c",
        "SaltOrd": "c", "CoastDist_m": "c", "Elevation_m": "c", "HumanUse": "b", "MechDamage": "b",
        "Flooding": "b", "Flowering": "b", "Fruiting": "b"}
for lu in trees.LandUse.unique(): df[f"LU_{lu.replace(' ', '_')}"] = (df.LandUse == lu).astype(int); EXPO[f"LU_{lu.replace(' ', '_')}"] = "b"
for so in trees.Soil.unique(): df[f"Soil_{so.replace(' ', '_')}"] = (df.Soil == so).astype(int); EXPO[f"Soil_{so.replace(' ', '_')}"] = "b"
CLIM = {"Rain2m", "TempC", "RH"}
OUTC = {"StressIdx": sm.families.Gaussian(), "PoorCanopy": sm.families.Binomial(),
        "Pest": sm.families.Binomial(), "Disease": sm.families.Binomial()}
ew = []
dz = df.copy()
for e_, t_ in EXPO.items():
    if t_ == "c": dz[e_] = (dz[e_] - dz[e_].mean()) / dz[e_].std()
for o_, fam in OUTC.items():
    for e_ in EXPO:
        if dz[e_].nunique() < 2: continue
        # guard against quasi-separation: binary exposures need >=10 exposed rows,
        # and for binary outcomes every 2x2 cell needs >=3 rows
        if EXPO[e_] == "b":
            if df[e_].sum() < 10: continue
            if isinstance(fam, sm.families.Binomial) and pd.crosstab(df[e_], df[o_]).values.min() < 3: continue
        form = f"{o_} ~ {e_}" + ("" if e_ in CLIM else " + WetSeason")
        try:
            r = smf.gee(form, "Tree_ID", dz, family=fam, cov_struct=sm.cov_struct.Exchangeable()).fit()
            b, se, p = r.params[e_], r.bse[e_], r.pvalues[e_]
        except Exception:
            b, se, p = np.nan, np.nan, np.nan
        ew.append({"outcome": o_, "exposure": e_, "beta": b, "se": se, "p": p,
                   "effect": "OR" if isinstance(fam, sm.families.Binomial) else "beta",
                   "estimate": np.exp(b) if isinstance(fam, sm.families.Binomial) else b})
ew = pd.DataFrame(ew).dropna(subset=["p"])
ew["q_BH"] = multipletests(ew.p, method="fdr_bh")[1]
ew.sort_values("p").round(5).to_csv(os.path.join(TAB, "T10_EWAS_GEE.csv"), index=False)
print(ew.sort_values("p").head(15).round(4))
SUMMARY["ewas_sig_q05"] = ew[ew.q_BH < 0.05][["outcome", "exposure", "estimate", "q_BH"]].round(4).to_dict("records")

fig, ax = plt.subplots(figsize=(10, 3.8)); x0 = 0; ticks = []; cols = ["#1565C0", "#6A1B9A", "#EF6C00", "#2E7D32"]
for k_, (o_, sub) in enumerate(ew.groupby("outcome", sort=False)):
    xs = np.arange(len(sub)) + x0
    ax.scatter(xs, -np.log10(sub.p), color=cols[k_], s=18, label=o_)
    for x_, (_, r) in zip(xs, sub.iterrows()):
        if r.q_BH < 0.05: ax.annotate(r.exposure, (x_, -np.log10(r.p)), fontsize=6, rotation=30, xytext=(2, 2), textcoords="offset points")
    ticks.append(xs.mean()); x0 += len(sub) + 3
thr = ew.loc[ew.q_BH < 0.05, "p"].max() if (ew.q_BH < 0.05).any() else 0.05 / len(ew)
ax.axhline(-np.log10(thr), ls="--", c="grey", lw=0.8); ax.text(x0 - 2, -np.log10(thr) + 0.1, "FDR 5%", fontsize=7, ha="right")
ax.set_xticks(ticks); ax.set_xticklabels(ew.outcome.unique()); ax.set_ylabel("-log10 p")
ax.set_title("EWAS 'Manhattan' plot — exposures × health outcomes (GEE, tree-clustered)")
savefig("F7_EWAS_manhattan.png")

# %% [markdown]
# ## M9. limma-style empirical-Bayes differential growth (coastal vs inland palms)
# Per-tree features: annual growth slopes (height, girth, DBH, crown) and mean leaf phenotypes.
# Moderated t (Smyth 2004): variance shrinkage with prior d0, s0² estimated by fitFDist.

# %%
def tree_slope(g, col):
    return np.polyfit(g["Years"], g[col], 1)[0]
feat = pd.DataFrame(index=ids)
for c in ["Height_m", "Girth_cm", "DBH_cm", "Crown_m"]:
    feat[f"growth_{c}"] = df.groupby("Tree_ID").apply(lambda g: tree_slope(g, c))
for c in ["LiveLeaves", "DeadLeaves", "NewLeaves", "Canopy", "Yellowing", "Drying", "StressIdx",
          "PestSev", "DiseaseSev", "Flowering", "Fruiting"]:
    feat[f"mean_{c}"] = df.groupby("Tree_ID")[c].mean()
feat.round(4).to_csv(os.path.join(TAB, "T11_tree_level_features.csv"))
grp = trees.loc[ids, "Coastal"].values
Fz = (feat - feat.mean()) / feat.std()

def fit_fdist(s2, d):
    z = np.log(s2); e = z - digamma(d / 2) + np.log(d / 2)
    emean = e.mean(); evar = e.var(ddof=1) - polygamma(1, d / 2)
    if evar <= 0: return np.inf, np.exp(emean)
    d0 = 2 * brentq(lambda x: polygamma(1, x) - evar, 1e-8, 1e8)
    s02 = np.exp(emean + digamma(d0 / 2) - np.log(d0 / 2)); return d0, s02

n1, n0 = grp.sum(), (1 - grp).sum(); d = n1 + n0 - 2
diff = Fz[grp == 1].mean() - Fz[grp == 0].mean()
s2 = (((Fz[grp == 1] - Fz[grp == 1].mean()) ** 2).sum() + ((Fz[grp == 0] - Fz[grp == 0].mean()) ** 2).sum()) / d
d0, s02 = fit_fdist(s2.values, d)
s2_post = (d0 * s02 + d * s2) / (d0 + d) if np.isfinite(d0) else pd.Series(s02, index=s2.index)
dft = d + (d0 if np.isfinite(d0) else 1e6)
t_mod = diff / np.sqrt(s2_post * (1 / n1 + 1 / n0))
p_mod = 2 * stats.t.sf(np.abs(t_mod), dft)
dg = pd.DataFrame({"diff_SD": diff, "mod_t": t_mod, "p": p_mod})
dg["q_BH"] = multipletests(dg.p, method="fdr_bh")[1]
dg = dg.sort_values("p"); dg.round(5).to_csv(os.path.join(TAB, "T12_limma_differential_growth.csv"))
SUMMARY["limma_prior"] = {"d0": float(d0) if np.isfinite(d0) else "inf", "s0^2": float(s02)}
print(dg.round(4))

fig, ax = plt.subplots(figsize=(5.5, 4))
c_ = np.where(dg.q_BH < 0.05, np.where(dg.diff_SD > 0, "#C62828", "#1565C0"), "#9E9E9E")
ax.scatter(dg.diff_SD, -np.log10(dg.p), c=c_, s=28)
for nm, r in dg.iterrows():
    if r.q_BH < 0.1: ax.annotate(nm.replace("mean_", "").replace("growth_", "Δ"), (r.diff_SD, -np.log10(r.p)), fontsize=7,
                                 xytext=(3, 3), textcoords="offset points")
ax.axhline(-np.log10(0.05), ls="--", c="grey", lw=0.8); ax.axvline(0, c="grey", lw=0.5)
ax.set_xlabel("Coastal − inland (SD units)"); ax.set_ylabel("-log10 moderated p")
ax.set_title("Volcano: empirical-Bayes differential phenotype"); savefig("F8_volcano_limma.png")

# %% [markdown]
# ## M10. Leakage-safe forecasting of next-visit canopy stress (and pest, as a negative control)
# Features at visit t: current scores, soil/site, HMM filtered state posterior (HMM refit on training
# trees only; filtering uses visits ≤ t only), and the known calendar flag for t+1.
# Grouped 5-fold CV by Tree_ID, repeated 5 times. Baselines: persistence and site/soil-only.

# %%
def forward_posteriors(model, order, frame):
    """Filtered P(state_t | residues_1..t) for every visit: uses only past/current visits (no look-ahead)."""
    out = []
    for _, g in frame.groupby("Tree_ID", sort=False):
        x = g["ResCode"].values.reshape(-1, 1)
        out.extend(model.predict_proba(x[:t + 1])[-1][order] for t in range(len(x)))
    return np.array(out)

df["next_Poor"] = df.groupby("Tree_ID")["PoorCanopy"].shift(-1)
df["next_Pest"] = df.groupby("Tree_ID")["Pest"].shift(-1)
df["next_Wet"] = (df["Observation_Date"] + pd.DateOffset(months=2)).dt.month.isin([11, 1]).astype(int)  # calendar is known in advance
SITE = ["SoilEC", "SoilPH", "SOC", "MoistOrd", "SaltOrd", "CoastDist_m", "Elevation_m", "HumanUse"] + \
       [c for c in df.columns if c.startswith("LU_") or c.startswith("Soil_")]
CUR = ["Canopy", "Yellowing", "Drying", "DeadLeaves", "LiveLeaves", "NewLeaves", "StressIdx", "Pest", "Disease",
       "Rain2m", "TempC", "RH", "WetSeason", "next_Wet"]

def run_cv(target, n_rep=5):
    res = []
    for rep in range(n_rep):
        perm_trees = rng.permutation(ids); fold_of = {t: i % 5 for i, t in enumerate(perm_trees)}
        folds = df["Tree_ID"].map(fold_of).values
        preds = {k: np.full(len(df), np.nan) for k in ["persistence", "site_only", "LR_full", "RF_full"]}
        for f_ in range(5):
            tr_t = [t for t in ids if fold_of[t] != f_]
            trm, tem = folds != f_, folds == f_
            dtr, dte = df[trm], df[tem]
            hm, hord, _ = fit_hmm(dtr, NS, seed=SEED + rep, restarts=4)
            Ptr, Pte = forward_posteriors(hm, hord, dtr), forward_posteriors(hm, hord, dte)
            hcols = [f"Z{i}" for i in range(NS)]
            Xtr = pd.concat([dtr[SITE + CUR].reset_index(drop=True), pd.DataFrame(Ptr, columns=hcols)], axis=1)
            Xte = pd.concat([dte[SITE + CUR].reset_index(drop=True), pd.DataFrame(Pte, columns=hcols)], axis=1)
            ytr = dtr[target].values; mtr = ~np.isnan(ytr); mte = ~np.isnan(dte[target].values)
            te_idx = np.where(tem)[0]
            base = "PoorCanopy" if target == "next_Poor" else "Pest"
            preds["persistence"][te_idx] = dte[base].values
            lr_s = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))
            lr_s.fit(Xtr.loc[mtr, SITE], ytr[mtr]); preds["site_only"][te_idx] = lr_s.predict_proba(Xte[SITE])[:, 1]
            lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))
            lr.fit(Xtr[mtr], ytr[mtr]); preds["LR_full"][te_idx] = lr.predict_proba(Xte)[:, 1]
            rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=5, random_state=SEED + rep, n_jobs=-1)
            rf.fit(Xtr[mtr], ytr[mtr]); preds["RF_full"][te_idx] = rf.predict_proba(Xte)[:, 1]
        y = df[target].values; m = ~np.isnan(y)
        for k_, p_ in preds.items():
            res.append({"target": target, "rep": rep, "model": k_, "AUROC": roc_auc_score(y[m], p_[m]),
                        "BalAcc": balanced_accuracy_score(y[m], p_[m] >= 0.5),
                        "Brier": brier_score_loss(y[m], np.clip(p_[m], 0, 1))})
    return pd.DataFrame(res)

cv = pd.concat([run_cv("next_Poor"), run_cv("next_Pest")])
cvs = cv.groupby(["target", "model"])[["AUROC", "BalAcc", "Brier"]].agg(["mean", "std"]).round(3)
cvs.to_csv(os.path.join(TAB, "T13_forecast_cv.csv")); print(cvs)
SUMMARY["forecast"] = {f"{a}|{b}": {"AUROC": float(cvs.loc[(a, b), ("AUROC", "mean")]),
                                    "AUROC_sd": float(cvs.loc[(a, b), ("AUROC", "std")])} for a, b in cvs.index}

SUMMARY["forecast_note"] = "AUROC averaged over 5 repeats of tree-grouped 5-fold CV"

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, tg, ttl in zip(axes, ["next_Poor", "next_Pest"], ["Next-visit poor canopy", "Next-visit pest (negative control)"]):
    sub = cv[cv.target == tg]
    sns.barplot(sub, x="model", y="AUROC", ax=ax, color="#5C6BC0", errorbar="sd",
                order=["persistence", "site_only", "LR_full", "RF_full"])
    ax.axhline(0.5, ls="--", c="grey", lw=0.8); ax.set_title(ttl); ax.set_xlabel(""); ax.set_ylim(0.3, 1)
    ax.tick_params(axis="x", rotation=20)
savefig("F9_forecast_auroc.png")

# Feature importance (RF, full data, descriptive only)
Pall = forward_posteriors(hmm, order_s, df)
Xall = pd.concat([df[SITE + CUR].reset_index(drop=True), pd.DataFrame(Pall, columns=[f"Z{i}" for i in range(NS)])], axis=1)
mm = df["next_Poor"].notna().values
rf = RandomForestClassifier(n_estimators=600, min_samples_leaf=5, random_state=SEED, n_jobs=-1).fit(Xall[mm], df.loc[mm, "next_Poor"])
from sklearn.inspection import permutation_importance
pi = permutation_importance(rf, Xall[mm], df.loc[mm, "next_Poor"], n_repeats=20, random_state=SEED, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=Xall.columns).sort_values(ascending=False)
imp.round(4).to_csv(os.path.join(TAB, "T14_permutation_importance.csv"), header=["dAUROC"])
fig, ax = plt.subplots(figsize=(5, 4)); imp.head(12)[::-1].plot.barh(ax=ax, color="#26A69A")
ax.set_xlabel("Permutation importance (ΔAUROC)"); ax.set_title("Drivers of next-visit canopy stress")
savefig("F10_importance.png")

# %%
with open(os.path.join(OUT, "summary.json"), "w") as f:
    json.dump(SUMMARY, f, indent=2, default=str)
print(json.dumps(SUMMARY, indent=2, default=str))
print("\nDone. Figures ->", FIG, " Tables ->", TAB)

50 trees x 9 visits = 450 observations
PALM-BLOSUM:
    H  L  M  S
H  1  0 -1 -3
L  0  0  0 -2
M -1  0  0  1
S -3 -2  1  4
        n  coast_m  salt  elev  mean_stress consensus_seq
Clade                                                    
1       5  2007.40  0.60  6.30         3.67     LMMMLHHLM
2       6  2921.17  0.00  6.70         2.15     MLHHLHHLM
3       6  1615.17  1.00  6.30         3.52     MHMMHHMHM
4      11  2357.91  0.45  7.05         3.21     MLMMHMMMM
5       8  2594.50  0.25  5.31         2.97     HLLMHHMML
6      14  2183.00  0.43  6.21         2.21     HMMHHHMHH
         factor  pseudo_F     R2  p_perm    q_BH
0       LandUse     1.170  0.117  0.3950  0.7353
1          Soil     1.047  0.064  0.4490  0.7353
2  SaltExposure     0.835  0.034  0.5515  0.7353
3  CoastTertile     0.005  0.000  0.8430  0.8430
    k motif  coastal_with  coastal_n  inland_with  inland_n  odds_ratio  \
10  2    MS            10         18            1        32   38.750000   
47  3   MSM       